In [2]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path('~/work/ERP/emt/src').expanduser()))

# Add src to path
REPO_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
SRC_DIR = REPO_ROOT / 'src'
#sys.path.insert(0, str(SRC_DIR))
OUTPUTS_DIR = Path(REPO_ROOT, "user-data/outputs")
# Data directory
DATA_DIR = Path(REPO_ROOT,'data/wellness_centre')

print(f"Repository: {REPO_ROOT}")
print(f"Source: {SRC_DIR}")
print(f"Data: {DATA_DIR}")
print(f"Outputs: {OUTPUTS_DIR}")
print("✓ Path configured")

Repository: /home/jovyan/work/ERP/emt
Source: /home/jovyan/work/ERP/emt/src
Data: /home/jovyan/work/ERP/emt/data/wellness_centre
Outputs: /home/jovyan/work/ERP/emt/user-data/outputs
✓ Path configured


In [3]:
# Cell 2: Connect to ERPNext (Production Configuration)

from frappeclient import FrappeClient
import os
import csv
from pathlib import Path

# ─────────────────────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────────────────────

# Internal Docker URL — avoids Cloudflare round-trip latency
ERPNEXT_URL = "http://erpnext-frontend:8080"
ERPNEXT_DOMAIN = "well.rosslyn.cloud"  # Host header for internal routing

# Path to API credentials file
# CSV format: two columns named 'api_key' and 'api_secret', one row of values
API_KEYS_FILE = Path("/home/jovyan/work/ERP/Wellness Centre/frappe_api_keys.csv")

# ─────────────────────────────────────────────────────────────
# Credential Loading (CSV → Environment → Manual)
# ─────────────────────────────────────────────────────────────

API_KEY = ""
API_SECRET = ""

if API_KEYS_FILE.exists():
    with open(API_KEYS_FILE, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            API_KEY = row.get("api_key", "").strip()
            API_SECRET = row.get("api_secret", "").strip()
            break  # Only read first data row
    print(f"✓ Credentials loaded from {API_KEYS_FILE}")
else:
    # Fall back to environment variables
    API_KEY = os.environ.get("ERPNEXT_API_KEY", "")
    API_SECRET = os.environ.get("ERPNEXT_API_SECRET", "")
    
    if API_KEY:
        print("✓ Credentials loaded from environment variables")
    else:
        print(f"⚠ Credentials file not found: {API_KEYS_FILE}")
        print("  Set ERPNEXT_API_KEY / ERPNEXT_API_SECRET environment variables,")
        print("  or create the CSV file, or paste keys directly below:")
        print()
        print("  API_KEY = 'your_api_key_here'")
        print("  API_SECRET = 'your_api_secret_here'")

# ─────────────────────────────────────────────────────────────
# Validation
# ─────────────────────────────────────────────────────────────

problems = []

if not API_KEY:
    problems.append("API_KEY is empty")
if not API_SECRET:
    problems.append("API_SECRET is empty")
if not DATA_DIR.exists():
    problems.append(f"DATA_DIR not found: {DATA_DIR}")

if problems:
    print("\n⚠ Configuration problems:")
    for p in problems:
        print(f"  • {p}")
    raise ValueError("Configuration incomplete - fix issues above before continuing")
else:
    print(f"\n✓ Configuration validated")
    print(f"  URL:      {ERPNEXT_URL}")
    print(f"  Domain:   {ERPNEXT_DOMAIN}")
    print(f"  Data dir: {DATA_DIR}")
    print(f"  API key:  {'set (' + API_KEY[:8] + '...)' if API_KEY else 'MISSING'}")

# ─────────────────────────────────────────────────────────────
# Connect to ERPNext
# ─────────────────────────────────────────────────────────────

print("\nConnecting to ERPNext...")

try:
    # Create client
    client = FrappeClient(ERPNEXT_URL)
    
    # Authenticate
    client.authenticate(API_KEY, API_SECRET)
    
    # CRITICAL: Add Host header for Docker network routing
    client.session.headers.update({"Host": ERPNEXT_DOMAIN})
    
    # Test connection
    test = client.get_list("Customer", fields=["name"], limit_page_length=1)
    
    print(f"✓ Connected to ERPNext at {ERPNEXT_URL}")
    print(f"  Host header: {ERPNEXT_DOMAIN}")
    print(f"  Status: Authenticated and working")
    print()
    
except Exception as e:
    print(f"✗ Connection failed: {e}")
    print()
    print("Troubleshooting:")
    print("  1. Check ERPNext is running: docker ps | grep erpnext")
    print("  2. Verify API credentials are correct")
    print("  3. Try external URL as fallback:")
    print("     ERPNEXT_URL = 'https://well.rosslyn.cloud'")
    print("     # Remove Host header line")
    print()
    raise

✓ Credentials loaded from /home/jovyan/work/ERP/Wellness Centre/frappe_api_keys.csv

✓ Configuration validated
  URL:      http://erpnext-frontend:8080
  Domain:   well.rosslyn.cloud
  Data dir: /home/jovyan/work/ERP/emt/data/wellness_centre
  API key:  set (c7edf88e...)

Connecting to ERPNext...
✓ Connected to ERPNext at http://erpnext-frontend:8080
  Host header: well.rosslyn.cloud
  Status: Authenticated and working



In [4]:
# Verify installation
try:
    from orchestration.csv_based_master_data import CSVBasedMasterDataCreator
    from orchestration.etims_invoice_importer import EtimsInvoiceImporter
    print("✓ Modules imported successfully")
except ImportError as e:
    print(f"✗ Import failed: {e}")
    print("\nRun the bash commands above first")

✓ Modules imported successfully


In [5]:
# Create master data from ACTUAL CSV files
csv_creator = CSVBasedMasterDataCreator(
    client=client,
    company="Wellness Centre",
    data_dir=Path(DATA_DIR)
)

# Create EVERYTHING (UOMs, Items, Customers)
results = csv_creator.create_all_from_csv()

# Check results
print("\n" + "="*70)
all_success = (
    results['uoms']['created'] == results['uoms']['total'] and
    results['items']['created'] == results['items']['total'] and
    results['customers']['created'] == results['customers']['total']
)

if all_success:
    print("✓✓✓ COMPLETE SUCCESS!")
    print()
    print("All master data created:")
    print(f"  ✓ UOMs:      {results['uoms']['created']}")
    print(f"  ✓ Items:     {results['items']['created']}")
    print(f"  ✓ Customers: {results['customers']['created']}")
    print()
    print("Ready to import invoices!")
else:
    print("⚠ Some items failed")
    print(f"  UOMs:      {results['uoms']['created']}/{results['uoms']['total']}")
    print(f"  Items:     {results['items']['created']}/{results['items']['total']}")
    print(f"  Customers: {results['customers']['created']}/{results['customers']['total']}")
    if results['errors']:
        print(f"  Errors: {len(results['errors'])}")

print("="*70)

CREATING MASTER DATA FROM CSV FILES

1. Analyzing items from etims_invoice_items.csv...
   Found 10 unique items

2. Creating required UOMs...
   ✓ Event
   ✓ Night
   ✓ Session
   ✓ Tray

3. Creating items in ERPNext...
   ✓ Fresh farm eggs (tray of 30)
   ✓ Event venue hire — deposit (50%)
   ✓ Event venue hire — balance payment
   ✓ Guest Bedroom — nightly rate
   ✓ Wellness service — yoga session
   ✓ Master Bedroom — nightly rate
   ✓ Wellness service — nature therapy walk
   ✓ Wellness service — meditation class
   ✓ Wellness service — sound healing
   ✓ Wellness service — wellness consultation

4. Analyzing customers from etims_invoices.csv...
   Found 12 unique customers

5. Creating customers in ERPNext...
   ✓ Alice Kahiga
   ✓ Apiyo Kibe
   ✓ Atieno Ndirangu
   ✓ Damaris Sigei
   ✓ Dorothy Barasa
   ✓ Juma Korir
   ✓ Kiprono Muhoro
   ✓ Lagat Nyambura
   ✓ Maina Mundia
   ✓ Murimi Onyango
   ... and 2 more

MASTER DATA CREATION COMPLETE
UOMs:      4/4
Items:     10/10
Custom

In [17]:
# Cell 5: Import eTIMS Invoices (TEST with 10 first)
from orchestration.etims_invoice_importer import EtimsInvoiceImporter
from pathlib import Path

importer = EtimsInvoiceImporter(
    client=client,
    data_dir=Path(DATA_DIR)
)

# Test with 10 invoices first
print("Testing with 10 invoices...")
test_result = importer.import_all(
    check_duplicates=True,
    auto_submit=False,  # Keep as Draft
    limit=10
)

print(test_result.summary())

Testing with 10 invoices...
Loading eTIMS data from CSV...
Found 10 invoices to import

  Progress: 10/10 (✓ 0, ⊘ 0, ✗ 10)

IMPORT SUMMARY — Sales Invoice (eTIMS)
  Total records:  10
  Succeeded:      0
  Skipped:        0  (already existed)
  Failed:         10
  Duration:       10s

FAILURES (10):
  - [Damaris Sigei] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [Apiyo Kibe] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [Dorothy Barasa] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [Apiyo Kibe] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [Atieno Ndirangu] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [Apiyo Kibe] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", lin

In [18]:
# Cell: Test invoice creation directly (bypass importer)
test_payload = {
    "doctype": "Sales Invoice",
    "customer": "Damaris Sigei",
    "customer_name": "Damaris Sigei",
    "posting_date": "2024-03-01",
    "due_date": "2024-03-01",
    "items": [
        {
            "item_code": "Fresh farm eggs (tray of 30)",
            "item_name": "Fresh farm eggs (tray of 30)",
            "description": "Fresh farm eggs (tray of 30)",
            "qty": 3.0,
            "rate": 345.0,
            "amount": 1035.0,
            "uom": "Tray"
        }
    ],
    "remarks": "Test invoice"
}

try:
    result = client.insert(test_payload)
    print("✓ SUCCESS!")
    print(f"Created: {result.get('name')}")
    
except Exception as e:
    print("✗ FAILED")
    print()
    print("Full error:")
    print(str(e))

✗ FAILED

Full error:
["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/api/__init__.py\", line 52, in handle\n    data = endpoint(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/api/v1.py\", line 46, in create_doc\n    return frappe.new_doc(doctype, **data).insert()\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/model/document.py\", line 309, in insert\n    self.run_before_save_methods()\n  File \"apps/frappe/frappe/model/document.py\", line 1140, in run_before_save_methods\n    self.run_method(\"validate\")\n  File \"apps/frappe/frappe/model/document.py\", line 1011, in run_method\n    out = Document.hook(fn)(self, *args, **kwargs)\n          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/model/document.py\", line 1371, in composer\n   

In [19]:
# Cell: Check how customer was created
try:
    customers = client.get_list(
        "Customer",
        filters={"customer_name": "Damaris Sigei"},
        fields=["name", "customer_name"],
        limit_page_length=5
    )
    
    if customers:
        print("✓ Customer found:")
        for c in customers:
            print(f"  name (ID): {c['name']}")
            print(f"  customer_name: {c['customer_name']}")
    else:
        print("✗ Customer 'Damaris Sigei' NOT FOUND")
        
        # Check all customers
        all_customers = client.get_list(
            "Customer",
            fields=["name", "customer_name"],
            limit_page_length=20
        )
        print(f"\nAll customers ({len(all_customers)}):")
        for c in all_customers:
            print(f"  {c['name']} → {c['customer_name']}")
            
except Exception as e:
    print(f"Error: {e}")

✓ Customer found:
  name (ID): CUST-00039
  customer_name: Damaris Sigei


In [23]:
# Cell: Reload importer with fix
import sys
if 'orchestration.etims_invoice_importer' in sys.modules:
    del sys.modules['orchestration.etims_invoice_importer']

from orchestration.etims_invoice_importer import EtimsInvoiceImporter
from pathlib import Path

# Recreate importer
importer = EtimsInvoiceImporter(
    client=client,
    data_dir=Path(DATA_DIR)
)

print("✓ Importer reloaded with customer ID fix")

✓ Importer reloaded with customer ID fix


In [24]:
# Cell: Test again with 10 invoices
test_result = importer.import_all(
    check_duplicates=True,
    auto_submit=False,
    limit=10
)

print(test_result.summary())

Loading eTIMS data from CSV...
Found 10 invoices to import

  Progress: 10/10 (✓ 0, ⊘ 0, ✗ 10)

IMPORT SUMMARY — Sales Invoice (eTIMS)
  Total records:  10
  Succeeded:      0
  Skipped:        0  (already existed)
  Failed:         10
  Duration:       21s

FAILURES (10):
  - [Damaris Sigei] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [Apiyo Kibe] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [Dorothy Barasa] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [Apiyo Kibe] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [Atieno Ndirangu] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [Apiyo Kibe] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application
  - [A

In [25]:
# Cell: Check what's missing in ERPNext setup
print("CHECKING ERPNEXT PREREQUISITES")
print("="*70)

# 1. Check Income Account
try:
    income_accounts = client.get_list(
        "Account",
        filters={
            "company": "Wellness Centre",
            "account_type": "Income Account"
        },
        fields=["name", "account_name"],
        limit_page_length=5
    )
    print(f"✓ Income Accounts: {len(income_accounts)}")
    if income_accounts:
        for acc in income_accounts[:3]:
            print(f"  - {acc['name']}")
except Exception as e:
    print(f"✗ Income Accounts: {e}")

print()

# 2. Check Debtors Account
try:
    debtors = client.get_list(
        "Account",
        filters={
            "company": "Wellness Centre",
            "account_type": "Receivable"
        },
        fields=["name", "account_name"],
        limit_page_length=5
    )
    print(f"✓ Debtors/Receivable Accounts: {len(debtors)}")
    if debtors:
        for acc in debtors[:3]:
            print(f"  - {acc['name']}")
except Exception as e:
    print(f"✗ Debtors Accounts: {e}")

print()

# 3. Check Mode of Payment
try:
    payment_modes = client.get_list(
        "Mode of Payment",
        fields=["name"],
        limit_page_length=10
    )
    print(f"✓ Mode of Payment: {len(payment_modes)}")
    if payment_modes:
        for mode in payment_modes:
            print(f"  - {mode['name']}")
    else:
        print("  ✗ NO MODES OF PAYMENT FOUND!")
except Exception as e:
    print(f"✗ Mode of Payment: {e}")

print()

# 4. Check Company default accounts
try:
    company = client.get_doc("Company", "Wellness Centre")
    print("✓ Company Default Accounts:")
    print(f"  Default Income Account: {company.get('default_income_account', 'NOT SET')}")
    print(f"  Default Receivable Account: {company.get('default_receivable_account', 'NOT SET')}")
    print(f"  Default Cash Account: {company.get('default_cash_account', 'NOT SET')}")
except Exception as e:
    print(f"✗ Company: {e}")

print()

# 5. Check Customer default settings
try:
    customer = client.get_doc("Customer", "CUST-00039")
    print("✓ Customer (Damaris Sigei) Settings:")
    print(f"  Default Currency: {customer.get('default_currency', 'NOT SET')}")
    print(f"  Payment Terms: {customer.get('payment_terms', 'NOT SET')}")
    print(f"  Credit Limit: {customer.get('credit_limit', 'NOT SET')}")
except Exception as e:
    print(f"✗ Customer Settings: {e}")

print("="*70)

CHECKING ERPNEXT PREREQUISITES
✓ Income Accounts: 0

✓ Debtors/Receivable Accounts: 1
  - Debtors - WC

✓ Mode of Payment: 7
  - Cheque
  - Credit Card
  - Wire Transfer
  - Bank Draft
  - Cash
  - M-Pesa
  - Bank Transfer

✓ Company Default Accounts:
  Default Income Account: Sales - WC
  Default Receivable Account: Debtors - WC
  Default Cash Account: Cash - WC

✓ Customer (Damaris Sigei) Settings:
  Default Currency: NOT SET
  Payment Terms: NOT SET
  Credit Limit: NOT SET


In [26]:
# Cell: Check current state
print("CURRENT ERPNEXT STATE")
print("="*70)

# Check customers
customers = client.get_list(
    "Customer",
    fields=["name", "customer_name"],
    limit_page_length=20
)
print(f"✓ Customers: {len(customers)}")
for c in customers[:5]:
    print(f"  {c['name']} → {c['customer_name']}")

print()

# Check items
items = client.get_list(
    "Item",
    filters={"is_stock_item": 0},
    fields=["item_code"],
    limit_page_length=15
)
print(f"✓ Service Items: {len(items)}")
for item in items[:5]:
    print(f"  {item['item_code']}")

print()

# Check existing invoices
existing_invoices = client.get_list(
    "Sales Invoice",
    fields=["name", "customer", "grand_total", "docstatus"],
    limit_page_length=5
)
print(f"✓ Existing Invoices: {len(existing_invoices)}")

print("="*70)

CURRENT ERPNEXT STATE
✓ Customers: 20
  CUST-00034 → Alice Kahiga
  CUST-00041 → Apiyo Kibe
  CUST-00040 → Atieno Ndirangu
  Beatrice Mwende → Beatrice Mwende
  Caroline Muthama → Caroline Muthama

✓ Service Items: 15
  ACCOMMODATION
  EGGS
  Event Venue Hire
  Event venue hire — balance payment
  Event venue hire — deposit (50%)

✓ Existing Invoices: 5


In [27]:
# Cell: Check existing invoices
existing_invoices = client.get_list(
    "Sales Invoice",
    fields=["name", "customer", "customer_name", "grand_total", "posting_date", "docstatus"],
    limit_page_length=10
)

print("EXISTING INVOICES:")
print("="*70)
for inv in existing_invoices:
    status = "Draft" if inv['docstatus'] == 0 else "Submitted"
    print(f"{inv['name']}: {inv['customer_name']}")
    print(f"  Customer ID: {inv['customer']}")
    print(f"  Date: {inv['posting_date']}")
    print(f"  Total: {inv['grand_total']}")
    print(f"  Status: {status}")
    print()

print("="*70)

EXISTING INVOICES:
ACC-SINV-2026-00206: Damaris Sigei
  Customer ID: CUST-00039
  Date: 2026-02-25
  Total: 1035.0
  Status: Submitted

ACC-SINV-2026-00207: Apiyo Kibe
  Customer ID: CUST-00041
  Date: 2026-02-25
  Total: 2720.0
  Status: Submitted

ACC-SINV-2026-00208: Apiyo Kibe
  Customer ID: CUST-00041
  Date: 2026-02-25
  Total: 2070.0
  Status: Submitted

ACC-SINV-2026-00209: Atieno Ndirangu
  Customer ID: CUST-00040
  Date: 2026-02-25
  Total: 720.0
  Status: Submitted

ACC-SINV-2026-00210: Apiyo Kibe
  Customer ID: CUST-00041
  Date: 2026-02-25
  Total: 2070.0
  Status: Submitted

ACC-SINV-2026-00211: Atieno Ndirangu
  Customer ID: CUST-00040
  Date: 2026-02-25
  Total: 700.0
  Status: Submitted

ACC-SINV-2026-00212: Murimi Onyango
  Customer ID: CUST-00042
  Date: 2026-02-25
  Total: 2100.0
  Status: Submitted

ACC-SINV-2026-00213: Murimi Onyango
  Customer ID: CUST-00042
  Date: 2026-02-25
  Total: 2485.0
  Status: Submitted

ACC-SINV-2026-00214: Damaris Sigei
  Customer ID: 

In [31]:
# Cell: Test with 10 invoices
import sys
from pathlib import Path

# Clear cache
if 'orchestration.etims_invoice_importer' in sys.modules:
    del sys.modules['orchestration.etims_invoice_importer']

# Reimport
sys.path.insert(0, str(Path('~/work/ERP/emt/src').expanduser()))
from orchestration.etims_invoice_importer import EtimsInvoiceImporter

# Initialize
importer = EtimsInvoiceImporter(
    client=client,
    data_dir=Path(DATA_DIR)
)

print("Testing with 10 invoices...")
print("="*70)
print()

# Test
test_result = importer.import_all(
    check_duplicates=True,
    auto_submit=False,
    limit=10
)

# Print summary
importer.print_summary(test_result)

# Show first failure if any
if test_result['failed'] > 0:
    print("\nFIRST FAILURE DETAILS:")
    print("="*70)
    first = test_result['failures'][0]
    print(f"Customer: {first['customer']}")
    print(f"Date: {first['invoice_date']}")
    print()
    print("Full Error:")
    print(first['error'])
    print("="*70)


Testing with 10 invoices...

Loading eTIMS data from CSV...
Found 10 invoices to import

  Progress: 10/10 (✓ 0, ⊘ 0, ✗ 10)

IMPORT SUMMARY — Sales Invoice (eTIMS)
  Total records:  10
  Succeeded:      0
  Skipped:        0  (already existed)
  Failed:         10

FAILURES (10):
  - [Damaris Sigei] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n     
  - [Apiyo Kibe] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n     
  - [Dorothy Barasa] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n     
  - [Apiyo Kibe] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n     
  - [Atieno Ndirangu] ["Traceback (most 

In [34]:
# Cell: Analyze actual payment terms from CSV
import pandas as pd
from datetime import datetime

invoices_df = pd.read_csv('~/work/ERP/emt/data/wellness_centre/etims_invoices.csv')
txns_df = pd.read_csv('~/work/ERP/emt/data/wellness_centre/transactions.csv')
contacts_df = pd.read_csv('~/work/ERP/emt/data/wellness_centre/contacts.csv')

# Get payments
payments = txns_df[txns_df['type'] == 'income'].copy()

# Merge to get customer names
payments = payments.merge(
    contacts_df[['id', 'name']],
    left_on='contact_id',
    right_on='id',
    how='left'
)

# Merge with invoices
analysis = invoices_df.merge(
    payments,
    left_on=['customer_name', 'total_amount'],
    right_on=['name', 'amount'],
    how='inner',
    suffixes=('_inv', '_pmt')
)

# Calculate days between invoice and payment
analysis['invoice_date'] = pd.to_datetime(analysis['invoice_date'])
analysis['payment_date'] = pd.to_datetime(analysis['transaction_date'])
analysis['days_to_pay'] = (analysis['payment_date'] - analysis['invoice_date']).dt.days

print("ACTUAL PAYMENT TERMS ANALYSIS")
print("="*70)
print(f"Total matched: {len(analysis)} invoices")
print()
print("Days between invoice and payment:")
print(analysis['days_to_pay'].describe())
print()
print("Distribution:")
print(analysis['days_to_pay'].value_counts().sort_index().head(10))
print("="*70)

ACTUAL PAYMENT TERMS ANALYSIS
Total matched: 338 invoices

Days between invoice and payment:
count    338.000000
mean       0.000000
std       69.998601
min     -336.000000
25%        0.000000
50%        0.000000
75%        0.000000
max      336.000000
Name: days_to_pay, dtype: float64

Distribution:
days_to_pay
-336    1
-242    1
-228    1
-224    2
-203    2
-182    1
-172    2
-161    2
-140    1
-139    1
Name: count, dtype: int64


In [40]:
# Cell: Test with YOUR paths
import shutil
from pathlib import Path
import sys

# Your paths
REPO_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
SRC_DIR = REPO_ROOT / 'src'
DATA_DIR = Path(REPO_ROOT, 'data/wellness_centre')

# Clear cache
if 'orchestration.etims_invoice_importer' in sys.modules:
    del sys.modules['orchestration.etims_invoice_importer']

# Add to path
sys.path.insert(0, str(SRC_DIR))

# Import
from orchestration.etims_invoice_importer import EtimsInvoiceImporter

# Test
importer = EtimsInvoiceImporter(
    client=client,
    data_dir=DATA_DIR  # Use YOUR data path
)

test_result = importer.import_all(
    check_duplicates=True,
    auto_submit=False,
    limit=10
)

importer.print_summary(test_result)

Loading eTIMS data from CSV...
Found 10 invoices to import

  Progress: 10/10 (✓ 0, ⊘ 0, ✗ 10)

IMPORT SUMMARY — Sales Invoice (eTIMS)
  Total records:  10
  Succeeded:      0
  Skipped:        0  (already existed)
  Failed:         10

FAILURES (10):
  - [Damaris Sigei] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n     
  - [Apiyo Kibe] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n     
  - [Dorothy Barasa] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n     
  - [Apiyo Kibe] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n     
  - [Atieno Ndirangu] ["Traceback (most recent call last):\n  File \"

In [40]:
# Cell: Test with YOUR paths
import shutil
from pathlib import Path
import sys

# Your paths
REPO_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
SRC_DIR = REPO_ROOT / 'src'
DATA_DIR = Path(REPO_ROOT, 'data/wellness_centre')

# Clear cache
if 'orchestration.etims_invoice_importer' in sys.modules:
    del sys.modules['orchestration.etims_invoice_importer']

# Add to path
sys.path.insert(0, str(SRC_DIR))

# Import
from orchestration.etims_invoice_importer import EtimsInvoiceImporter

# Test
importer = EtimsInvoiceImporter(
    client=client,
    data_dir=DATA_DIR  # Use YOUR data path
)

test_result = importer.import_all(
    check_duplicates=True,
    auto_submit=False,
    limit=10
)

importer.print_summary(test_result)

Loading eTIMS data from CSV...
Found 10 invoices to import

  Progress: 10/10 (✓ 0, ⊘ 0, ✗ 10)

IMPORT SUMMARY — Sales Invoice (eTIMS)
  Total records:  10
  Succeeded:      0
  Skipped:        0  (already existed)
  Failed:         10

FAILURES (10):
  - [Damaris Sigei] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n     
  - [Apiyo Kibe] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n     
  - [Dorothy Barasa] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n     
  - [Apiyo Kibe] ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n     
  - [Atieno Ndirangu] ["Traceback (most recent call last):\n  File \"

In [41]:
# Cell: Get full error
if test_result['failed'] > 0:
    first = test_result['failures'][0]
    print("FULL ERROR:")
    print("="*70)
    print(f"Customer: {first['customer']}")
    print(f"Date: {first['invoice_date']}")
    print()
    print(first['error'])
    print("="*70)

FULL ERROR:
Customer: Damaris Sigei
Date: 2024-03-01

["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/api/__init__.py\", line 52, in handle\n    data = endpoint(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/api/v1.py\", line 46, in create_doc\n    return frappe.new_doc(doctype, **data).insert()\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/model/document.py\", line 309, in insert\n    self.run_before_save_methods()\n  File \"apps/frappe/frappe/model/document.py\", line 1140, in run_before_save_methods\n    self.run_method(\"validate\")\n  File \"apps/frappe/frappe/model/document.py\", line 1011, in run_method\n    out = Document.hook(fn)(self, *args, **kwargs)\n          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/model/document.p

In [42]:
# Cell: Verify the fix is in the file
with open(SRC_DIR / 'orchestration/etims_invoice_importer.py', 'r') as f:
    content = f.read()
    
# Check if fix is present
if 'timedelta(days=' in content:
    print("✓ Fix is in file")
    # Find the line
    for i, line in enumerate(content.split('\n')):
        if 'timedelta' in line:
            print(f"Line {i}: {line.strip()}")
else:
    print("✗ Fix NOT in file - need to update")
    
# Also check what due_date calculation looks like
print("\nSearching for due_date lines:")
for i, line in enumerate(content.split('\n')):
    if 'due_date' in line.lower():
        print(f"Line {i}: {line.strip()}")

✓ Fix is in file
Line 115: from datetime import datetime, timedelta
Line 120: due_dt = posting_dt + timedelta(days=15)

Searching for due_date lines:
Line 121: due_date = due_dt.strftime('%Y-%m-%d')
Line 128: "due_date": due_date,  # 15 days after posting


In [43]:
# Cell: Debug - see actual payload
import pandas as pd

# Load one invoice
invoices_df = pd.read_csv(DATA_DIR / 'etims_invoices.csv')
items_df = pd.read_csv(DATA_DIR / 'etims_invoice_items.csv')

first_invoice = invoices_df.iloc[0].to_dict()

# Build payload
payload = importer.build_erpnext_invoice(first_invoice, items_df)

# Show dates
print("PAYLOAD DATES:")
print(f"posting_date: {payload['posting_date']}")
print(f"due_date: {payload['due_date']}")
print(f"Type: {type(payload['due_date'])}")
print()

# Compare
from datetime import datetime
posting = datetime.strptime(payload['posting_date'], '%Y-%m-%d')
due = datetime.strptime(str(payload['due_date']), '%Y-%m-%d')

print(f"Posting: {posting}")
print(f"Due: {due}")
print(f"Difference: {(due - posting).days} days")

PAYLOAD DATES:
posting_date: 2024-03-01
due_date: 2024-03-16
Type: <class 'str'>

Posting: 2024-03-01 00:00:00
Due: 2024-03-16 00:00:00
Difference: 15 days


In [44]:
# Cell: Check customer configuration
customer_id = importer._get_customer_id("Damaris Sigei")
print(f"Customer ID: {customer_id}")
print()

try:
    customer = client.get_doc("Customer", customer_id)
    print("Customer Settings:")
    print(f"  payment_terms: {customer.get('payment_terms')}")
    print(f"  default_currency: {customer.get('default_currency')}")
    
    # If payment terms exist, check them
    if customer.get('payment_terms'):
        pt = client.get_doc("Payment Terms Template", customer['payment_terms'])
        print(f"\nPayment Terms Template: {pt.get('template_name')}")
        for term in pt.get('terms', []):
            print(f"  - {term}")
except Exception as e:
    print(f"Error: {e}")

Customer ID: CUST-00039

Customer Settings:
  payment_terms: None
  default_currency: None


In [45]:
# Cell: Test direct insert with explicit dates
test_payload = {
    "doctype": "Sales Invoice",
    "customer": customer_id,
    "posting_date": "2024-03-01",
    "due_date": "2024-03-16",
    "currency": "KES",
    "debit_to": "Debtors - WC",
    "items": [{
        "item_code": "Fresh farm eggs (tray of 30)",
        "qty": 3.0,
        "rate": 345.0,
        "income_account": "Sales - WC"
    }]
}

try:
    doc = client.insert(test_payload)
    print(f"✓ SUCCESS: {doc.get('name')}")
except Exception as e:
    print(f"✗ FAILED:")
    print(str(e)[:500])

✗ FAILED:
["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/api/__init__.py\", line 52, in handle\n    data = endpoint(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/api/v1.py\", line 46, in create_doc\n    return frappe.new_doc(doctype, **data).insert()\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File


In [46]:
# Cell: Check company fiscal year
company = client.get_doc("Company", "Wellness Centre")

print("Company Settings:")
print(f"  Fiscal Year: {company.get('default_fiscal_year')}")
print(f"  Country: {company.get('country')}")

# Check fiscal year dates
if company.get('default_fiscal_year'):
    fy = client.get_doc("Fiscal Year", company['default_fiscal_year'])
    print(f"\nFiscal Year Details:")
    print(f"  Name: {fy.get('name')}")
    print(f"  Year Start: {fy.get('year_start_date')}")
    print(f"  Year End: {fy.get('year_end_date')}")
    
    # Check if our invoice date is within fiscal year
    from datetime import datetime
    invoice_date = datetime.strptime("2024-03-01", "%Y-%m-%d")
    fy_start = datetime.strptime(str(fy.get('year_start_date')), "%Y-%m-%d")
    fy_end = datetime.strptime(str(fy.get('year_end_date')), "%Y-%m-%d")
    
    print(f"\nInvoice date (2024-03-01) within fiscal year: {fy_start <= invoice_date <= fy_end}")

Company Settings:
  Fiscal Year: None
  Country: Kenya


In [47]:
# Cell: Create Fiscal Year 2024
try:
    fy = client.insert({
        "doctype": "Fiscal Year",
        "year": "2024",
        "year_start_date": "2024-01-01",
        "year_end_date": "2024-12-31"
    })
    print(f"✓ Created Fiscal Year: {fy.get('name')}")
    
    # Set as company default
    client.update({
        "doctype": "Company",
        "name": "Wellness Centre",
        "default_fiscal_year": fy.get('name')
    })
    print(f"✓ Set as company default")
    
except Exception as e:
    print(f"Error: {e}")

Error: ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/api/__init__.py\", line 52, in handle\n    data = endpoint(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/api/v1.py\", line 46, in create_doc\n    return frappe.new_doc(doctype, **data).insert()\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/model/document.py\", line 334, in insert\n    self.run_post_save_methods()\n  File \"apps/frappe/frappe/model/document.py\", line 1177, in run_post_save_methods\n    self.run_method(\"on_update\")\n  File \"apps/frappe/frappe/model/document.py\", line 1011, in run_method\n    out = Document.hook(fn)(self, *args, **kwargs)\n          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/model/document.py\", line 1371, in composer\n    return composed(s

In [52]:
# Cell: Set existing fiscal year as default
client.update({
    "doctype": "Company",
    "name": "Wellness Centre",
    "default_fiscal_year": "FY 2024"
})

print("✓ Set FY 2024 as company default")

✓ Set FY 2024 as company default


In [53]:
# Cell: Test invoice after fiscal year
test_payload = {
    "doctype": "Sales Invoice",
    "customer": "CUST-00039",
    "posting_date": "2024-03-01",
    "due_date": "2024-03-16",
    "currency": "KES",
    "debit_to": "Debtors - WC",
    "items": [{
        "item_code": "Fresh farm eggs (tray of 30)",
        "qty": 3.0,
        "rate": 345.0,
        "income_account": "Sales - WC"
    }]
}

try:
    doc = client.insert(test_payload)
    print(f"✓ SUCCESS: {doc.get('name')}")
    print("Ready to import all 220 invoices!")
except Exception as e:
    print(f"✗ Still failing: {str(e)[:300]}")

✗ Still failing: ["Traceback (most recent call last):\n  File \"apps/frappe/frappe/app.py\", line 120, in application\n    response = frappe.api.handle(request)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"apps/frappe/frappe/api/__init__.py\", line 52, in handle\n    data = endpoint(**arguments)\n           


In [54]:
# Cell: Try without due_date (let ERPNext calculate it)
test_payload = {
    "doctype": "Sales Invoice",
    "customer": "CUST-00039",
    "posting_date": "2024-03-01",
    # Remove due_date - let ERPNext set it
    "currency": "KES",
    "debit_to": "Debtors - WC",
    "items": [{
        "item_code": "Fresh farm eggs (tray of 30)",
        "qty": 3.0,
        "rate": 345.0,
        "income_account": "Sales - WC"
    }]
}

try:
    doc = client.insert(test_payload)
    print(f"✓ SUCCESS: {doc.get('name')}")
except Exception as e:
    print(f"✗ Failed: {str(e)[:200]}")

✓ SUCCESS: ACC-SINV-2026-00427
